In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 1.11: Grand Comparison — All 10 Variants Side by Side
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Train ALL 10 autoencoder variants on the same data
#   - Compare reconstruction quality in a single mega-grid
#   - Compare training curves using ModelVisualizer
#   - Build a summary table with loss, parameters, and variant type
#   - Register all 10 models in ModelRegistry
#   - Choose the right variant for a given business problem
#
# PREREQUISITES: Files 01-10 (understanding each variant)
# ESTIMATED TIME: ~30 min (all 10 train sequentially)
#
# TASKS:
#   1. Train all 10 variants (reusing architectures from prior files)
#   2. Grand reconstruction comparison grid (9 image variants + original)
#   3. Training curves overlay
#   4. Summary table + model registration
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import pickle

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from kailash_ml import ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec



## TASK 1 — Load data and engines


In [ ]:
X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader = load_fashion_mnist()
conn, tracker, exp_name, registry, has_registry = setup_engines()

assert X_flat.shape[0] == 60000
assert X_test_flat.shape[0] == 10000
print("\n--- Data loaded ---\n")



## MODEL DEFINITIONS — All 10 Variants (compact)


In [ ]:
# These are the same architectures from files 01-10. You must implement
# each one here to train all 10 in a single comparison run.


class StandardAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=1024):
        super().__init__()
        # TODO: Overcomplete encoder/decoder (same as 01_standard_ae.py)
        self.encoder = ____
        self.decoder = ____

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class UndercompleteAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # TODO: Bottleneck encoder/decoder (same as 02_undercomplete_ae.py)
        self.encoder = ____
        self.decoder = ____

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class DenoisingAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # TODO: Same architecture as undercomplete (different training)
        self.encoder = ____
        self.decoder = ____

    def add_noise(self, x, sigma=0.3):
        return torch.clamp(x + sigma * torch.randn_like(x), 0.0, 1.0)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class SparseAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        # TODO: Sparse encoder/decoder (same as 04_sparse_ae.py)
        self.encoder = ____
        self.decoder = ____

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class ContractiveAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # TODO: Explicit encoder layers for weight access
        self.enc1 = ____
        self.enc2 = ____
        self.enc3 = ____
        self.decoder = ____

    def encoder(self, x):
        return self.enc3(F.relu(self.enc2(F.relu(self.enc1(x)))))

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class ConvAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        # TODO: Conv encoder + deconv decoder (same as 06_convolutional_ae.py)
        self.encoder = ____
        self.decoder = ____

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class StackedAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # TODO: 5-layer deep encoder/decoder (same as 07_stacked_ae.py)
        self.encoder = ____
        self.decoder = ____

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


class RecurrentAE(nn.Module):
    def __init__(self, seq_len, hidden_dim=64, latent_dim=16):
        super().__init__()
        self.seq_len = seq_len
        # TODO: LSTM encoder-decoder (same as 08_recurrent_ae.py)
        self.encoder_lstm = ____
        self.enc_to_latent = ____
        self.latent_to_dec = ____
        self.decoder_lstm = ____
        self.output_layer = ____

    def forward(self, x):
        # TODO: LSTM encode -> latent -> LSTM decode
        ____


class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # TODO: Encoder body + mu/logvar heads + decoder (same as 09_vae.py)
        self.encoder = ____
        self.fc_mu = ____
        self.fc_logvar = ____
        self.decoder = ____

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decoder(z), mu, logvar

    def sample(self, n):
        z = torch.randn(
            n, self.fc_mu.out_features, device=next(self.parameters()).device
        )
        return self.decoder(z)


class ContractiveVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        # TODO: Explicit encoder layers + mu/logvar + decoder
        self.enc1 = ____
        self.enc2 = ____
        self.fc_mu = ____
        self.fc_logvar = ____
        self.decoder = ____

    def encoder(self, x):
        return F.relu(self.enc2(F.relu(self.enc1(x))))

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return self.decoder(z), mu, logvar

    def sample(self, n):
        z = torch.randn(
            n, self.fc_mu.out_features, device=next(self.parameters()).device
        )
        return self.decoder(z)



## TASK 2 — Train All 10 Variants


In [ ]:
NOISE_SIGMA = 0.3
SPARSITY_WEIGHT = 1e-4
CONTRACTIVE_WEIGHT = 1e-4
KL_WEIGHT = 0.1
CVAE_CONTRACTIVE_WEIGHT = 1e-4


# TODO: Define loss functions for each variant
# std_loss: MSE only
# dae_loss: add noise, MSE against clean
# sparse_loss: MSE + L1 penalty
# cae_loss: MSE + Frobenius norm of encoder weights
# conv_loss: MSE only
# rec_loss: MSE only
# vae_loss: MSE + KL divergence
# cvae_loss: MSE + KL + Jacobian
def std_loss(m, xb):
    ____


def dae_loss(m, xb):
    ____


def sparse_loss(m, xb):
    ____


def cae_loss(m, xb):
    ____


def conv_loss(m, xb):
    ____


def rec_loss(m, xb):
    ____


def vae_loss(m, xb):
    ____


def cvae_loss(m, xb):
    ____


# Sensor data for recurrent AE
SEQ_LEN = 100
rng_s = np.random.RandomState(42)
sensor_data = []
for _ in range(5000):
    t = np.linspace(0, 4 * np.pi, SEQ_LEN)
    sig = rng_s.uniform(0.5, 1.0) * np.sin(
        rng_s.uniform(0.8, 1.2) * t + rng_s.uniform(0, 2 * np.pi)
    )
    sig += rng_s.uniform(0.2, 0.5) * np.sin(
        rng_s.uniform(1.8, 2.2) * t
    ) + 0.1 * rng_s.randn(SEQ_LEN)
    sig = (sig - sig.min()) / (sig.max() - sig.min() + 1e-8)
    sensor_data.append(sig)
sensor_t = torch.tensor(np.array(sensor_data, dtype=np.float32)).to(device)
sensor_loader = DataLoader(TensorDataset(sensor_t), batch_size=128, shuffle=True)

print("=" * 70)
print("  GRAND COMPARISON — Training All 10 Variants")
print("=" * 70)

# TODO: Create list of (name, model, loader, loss_fn) tuples for all 10 variants
# Then train each with train_variant
variants = [
    ("standard_ae", StandardAE(INPUT_DIM, 1024), flat_loader, std_loss),
    ("undercomplete_ae", UndercompleteAE(INPUT_DIM, LATENT_DIM), flat_loader, std_loss),
    ("denoising_ae", DenoisingAE(INPUT_DIM, LATENT_DIM), flat_loader, dae_loss),
    ("sparse_ae", SparseAE(INPUT_DIM), flat_loader, sparse_loss),
    ("contractive_ae", ContractiveAE(INPUT_DIM, LATENT_DIM), flat_loader, cae_loss),
    ("conv_ae", ConvAE(LATENT_DIM), img_loader, conv_loss),
    ("stacked_ae", StackedAE(INPUT_DIM, LATENT_DIM), flat_loader, std_loss),
    ("recurrent_ae", RecurrentAE(SEQ_LEN, 64, LATENT_DIM), sensor_loader, rec_loss),
    ("vae", VAE(INPUT_DIM, LATENT_DIM), flat_loader, vae_loss),
    ("contractive_vae", ContractiveVAE(INPUT_DIM, LATENT_DIM), flat_loader, cvae_loss),
]

for name, model, loader, loss_fn in variants:
    print(f"\n--- Training {name} ---")
    train_variant(tracker, exp_name, model, name, loader, loss_fn)

assert len(all_losses) == 10, f"Expected 10 variants, got {len(all_losses)}"
print(f"\n  All 10 variants trained successfully.")



## TASK 3 — Grand Reconstruction Comparison Grid


In [ ]:
image_variants = [
    ("Standard AE", all_models["standard_ae"], False),
    ("Undercomplete AE", all_models["undercomplete_ae"], False),
    ("Denoising AE", all_models["denoising_ae"], False),
    ("Sparse AE", all_models["sparse_ae"], False),
    ("Contractive AE", all_models["contractive_ae"], False),
    ("Convolutional AE", all_models["conv_ae"], True),
    ("Stacked AE", all_models["stacked_ae"], False),
    ("VAE", all_models["vae"], False),
    ("Contractive VAE", all_models["contractive_vae"], False),
]

N_COMPARE = 8
n_variants = len(image_variants)

# TODO: Create (n_variants + 1) x N_COMPARE grid figure
# Row 0: original images
# Rows 1-9: each variant's reconstruction
# Save to OUTPUT_DIR / "ex1_grand_comparison.png"
fig, axes = plt.subplots(
    n_variants + 1, N_COMPARE, figsize=(16, (n_variants + 1) * 1.5)
)
____
plt.tight_layout()
fname = OUTPUT_DIR / "ex1_grand_comparison.png"
plt.savefig(fname, dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: {fname}")



## TASK 4 — Summary Table


In [ ]:
name_to_key = {
    "Standard AE": "standard_ae",
    "Undercomplete AE": "undercomplete_ae",
    "Denoising AE": "denoising_ae",
    "Sparse AE": "sparse_ae",
    "Contractive AE": "contractive_ae",
    "Convolutional AE": "conv_ae",
    "Stacked AE": "stacked_ae",
    "VAE": "vae",
    "Contractive VAE": "contractive_vae",
}

# TODO: Print comparison table: Variant, Final Loss, Params, Type
print("\n=== Autoencoder Variant Comparison ===")
print(f"{'Variant':<22} {'Final Loss':>12} {'Params':>10} {'Type':>12}")
print("-" * 60)
for name, model, _ in image_variants:
    key = name_to_key[name]
    final_loss = all_losses[key][-1]
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {name:<20} {final_loss:>12.4f} {n_params:>10,} {'image':>12}")

rec_loss_val = all_losses["recurrent_ae"][-1]
rec_params = sum(p.numel() for p in all_models["recurrent_ae"].parameters())
print(
    f"  {'Recurrent AE':<20} {rec_loss_val:>12.4f} {rec_params:>10,} {'sequence':>12}"
)

# --- Training curves ---
# TODO: Use ModelVisualizer to plot all training curves
# Save to OUTPUT_DIR / "ex1_all_variants_training_curves.html"
viz = ModelVisualizer()
____



## MODEL REGISTRATION


In [ ]:
async def register_all():
    if not has_registry:
        print("  ModelRegistry not available — skipping")
        return
    for name, model in all_models.items():
        model_bytes = pickle.dumps(model.state_dict())
        version = await registry.register_model(
            name=f"m5_{name}",
            artifact=model_bytes,
            metrics=[
                MetricSpec(name="final_loss", value=all_losses[name][-1]),
                MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
                MetricSpec(name="epochs", value=float(EPOCHS)),
            ],
        )
        print(
            f"  Registered {name}: v{version.version}, loss={all_losses[name][-1]:.4f}"
        )


await register_all()



### Final Checkpoint


In [ ]:
assert len(all_losses) == 10
grand_path = OUTPUT_DIR / "ex1_grand_comparison.png"
assert grand_path.exists(), "Grand comparison image should be saved"
print("\n--- Final checkpoint passed --- grand comparison complete\n")

await conn.close()



## DESTINATION-FIRST CLOSE — km.diagnose


In [ ]:
# This lesson walked the journey of the autoencoder family — 10 variants,
# 10 hand-rolled training loops, 10 reconstruction grids. The kailash-ml
# SDK ships a single-call diagnostic primitive that closes the production
# loop: km.diagnose inspects a trained model and emits an auto-dashboard
# (loss curves, gradient flow, dead neurons, activation stats, weight
# distributions). One cell. Every diagnostic students would otherwise
# hand-roll, ready to surface in a Plotly dashboard.

from kailash_ml import diagnose

# Pick the VAE (the lesson's most expressive variant) for the close.
# `kind='auto'` dispatches by model type — DLDiagnostics for torch.nn.Module.
# `data=` accepts any iterable yielding tensors; we reuse the flat_loader
# the lesson already constructed.
report = diagnose(all_models["vae"], kind="auto", data=flat_loader, show=False)
report.plot_training_dashboard()
print()
print("km.diagnose: 1 line of code -> the same observability the lesson")
print("body hand-rolled in 200+ lines. This is what 'destination-first'")
print("means — when the journey is internalised, the SDK is one call.")



## REFLECTION


PROBLEM SOLVERS:
  [x] Standard AE: demonstrated the identity-function risk
  [x] Undercomplete AE: bottleneck forces 49:1 compression
  [x] Denoising AE: noise injection -> robust features
  [x] Sparse AE: L1 penalty -> specialist neurons

  ARCHITECTURE VARIANTS:
  [x] Contractive AE: Jacobian penalty -> smooth latent space
  [x] Convolutional AE: Conv2d -> sharpest image reconstructions
  [x] Stacked AE: depth -> hierarchical feature learning

  DIFFERENT MODALITIES:
  [x] Recurrent AE: LSTM -> temporal pattern capture

  GENERATIVE MODELS:
  [x] VAE: reparameterisation trick -> sample new data from N(0,I)
  [x] Contractive VAE: smooth + generative = best of both worlds

  ML ENGINEERING:
  [x] Tracked all 10 variants with ExperimentTracker
  [x] Registered all 10 models in ModelRegistry
  [x] Compared training curves with ModelVisualizer

  KEY INSIGHT: Every variant solves a specific failure mode:
  - Identity risk -> undercomplete bottleneck or regularisation
  - Memorisation -> noise injection (denoising) or sparsity
  - Sensitivity -> contractive penalty (Jacobian smoothness)
  - Generation need -> probabilistic latent space (VAE)
  - Spatial data -> convolutional architecture
  - Sequential data -> recurrent architecture

  Choosing the right variant is an engineering decision driven by
  your data type, use case, and failure tolerance.

  Next: Exercise 2 — CNNs with ResNet skip connections and SE
  attention for image classification...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — THE COMPLETE AUTOENCODER FAMILY")
print("=" * 70)
print(
)

